# 🧪 Sesión 14 · Hashing Seguro de Contraseñas (Bcrypt / Argon2)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué nunca se guarda una contraseña tal cual?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro  {"usuario":"ana","password":"secreta"}
HTTP/1.1 201 Created
{"usuario":"ana"}

POST /api/login  {"usuario":"ana","password":"secreta"}
HTTP/1.1 200 OK
{"usuario":"ana","autenticado":true}

POST /api/login  {"usuario":"ana","password":"incorrecta"}
HTTP/1.1 401 Unauthorized

La respuesta NUNCA contiene "password" ni el hash.
```

_El hash se guarda en el servidor; la contraseña original no se almacena en ningún lado._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Guardar contraseñas con bcrypt y salt.
- Verificar una contraseña sin descifrarla.
- Comprobar que la API nunca devuelve el hash ni la contraseña.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El registro responde sin filtrar la contraseña → HTTP 201
- [ ] El login correcto autentica → HTTP 200
- [ ] La contraseña incorrecta se rechaza → HTTP 401
- [ ] Se usa un algoritmo de hashing para contraseñas

## 🧩 El reto

Implementa registro y login con hashing seguro. La contraseña nunca debe viajar de vuelta al cliente.

**Requisitos del contrato**

- POST /api/registro recibe usuario y password, guarda solo el hash y responde 201.
- La respuesta de registro no incluye password ni hash.
- POST /api/login compara la contraseña con el hash y responde 200 o 401.
- Debes usar bcrypt (o passlib) para el hash.
- Un usuario repetido no debe sobrescribir la contraseña existente.

> **Pista conceptual:** No compares hashes con `==` si puedes usar la función de verificación del algoritmo. Bcrypt ya incluye el salt dentro del hash.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion14.md` explicando por qué no se puede «deshashear».


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, ConfigDict
from bcrypt import hashpw, gensalt, checkpw

# 1. Crear la aplicación FastAPI
app = FastAPI(
    title="API de Registro y Login Seguro",
    version="1.0.0"
)

# 2. Almacenamiento temporal de usuarios
# Aquí se guardará el nombre y el hash de cada contraseña.
usuarios_hash = {}


# 3. Modelo para recibir los datos del usuario
class UsuarioEntrada(BaseModel):
    model_config = ConfigDict(extra="forbid")

    usuario: str = Field(min_length=1)
    password: str = Field(min_length=1)


# 4. Modelo de respuesta del registro
class RegistroRespuesta(BaseModel):
    usuario: str


# 5. Modelo de respuesta del login
class LoginRespuesta(BaseModel):
    usuario: str
    autenticado: bool


# 6. Ruta para registrar usuarios
@app.post(
    "/api/registro",
    status_code=201,
    response_model=RegistroRespuesta
)
def registrar_usuario(datos: UsuarioEntrada):

    # Comprobar si el usuario ya existe
    if datos.usuario in usuarios_hash:
        raise HTTPException(
            status_code=409,
            detail="El usuario ya existe"
        )

    # Convertir la contraseña a bytes
    password_bytes = datos.password.encode("utf-8")

    # Generar un salt y crear el hash con Bcrypt
    password_hash = hashpw(
        password_bytes,
        gensalt()
    )

    # Guardar únicamente el hash
    usuarios_hash[datos.usuario] = password_hash

    # Responder sin mostrar contraseña ni hash
    return {
        "usuario": datos.usuario
    }


# 7. Ruta para iniciar sesión
@app.post(
    "/api/login",
    response_model=LoginRespuesta
)
def iniciar_sesion(datos: UsuarioEntrada):

    # Buscar el hash del usuario
    password_hash = usuarios_hash.get(datos.usuario)

    # Si el usuario no existe, rechazar el acceso
    if password_hash is None:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas"
        )

    # Verificar la contraseña con Bcrypt
    password_correcta = checkpw(
        datos.password.encode("utf-8"),
        password_hash
    )

    # Si la contraseña no coincide, rechazar el acceso
    if not password_correcta:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas"
        )

    # Si coincide, permitir el acceso
    return {
        "usuario": datos.usuario,
        "autenticado": True
    }


Writing app.py


In [5]:
from fastapi.testclient import TestClient
from app import app, usuarios_hash

# Crear cliente de pruebas
cliente = TestClient(app)

# Limpiar almacenamiento para comenzar desde cero
usuarios_hash.clear()

# PRUEBA 1: Registrar usuario
respuesta = cliente.post(
    "/api/registro",
    json={
        "usuario": "ana",
        "password": "secreta"
    }
)

print("REGISTRO:", respuesta.status_code)
print(respuesta.json())

assert respuesta.status_code == 201
assert respuesta.json() == {"usuario": "ana"}

# PRUEBA 2: Login con contraseña correcta
respuesta = cliente.post(
    "/api/login",
    json={
        "usuario": "ana",
        "password": "secreta"
    }
)

print("\nLOGIN CORRECTO:", respuesta.status_code)
print(respuesta.json())

assert respuesta.status_code == 200
assert respuesta.json()["autenticado"] == True

# PRUEBA 3: Login con contraseña incorrecta
respuesta = cliente.post(
    "/api/login",
    json={
        "usuario": "ana",
        "password": "incorrecta"
    }
)

print("\nLOGIN INCORRECTO:", respuesta.status_code)
print(respuesta.json())

assert respuesta.status_code == 401

# PRUEBA 4: Comprobar que no se devuelve el hash
assert "password" not in respuesta.json()
assert "hash" not in respuesta.json()

print("\n✅ TODAS LAS PRUEBAS MANUALES FUERON CORRECTAS")

REGISTRO: 201
{'usuario': 'ana'}

LOGIN CORRECTO: 200
{'usuario': 'ana', 'autenticado': True}

LOGIN INCORRECTO: 401
{'detail': 'Credenciales incorrectas'}

✅ TODAS LAS PRUEBAS MANUALES FUERON CORRECTAS


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Se trabajará con Python 3.13, FastAPI y Pydantic. Se utilizará Bcrypt para proteger las contraseñas mediante hashing seguro. La implementación se realizará en Google Colab sin necesidad de instalar herramientas localmente. |
| **Rol** | Actúa como desarrollador backend especializado en seguridad de aplicaciones web, autenticación de usuarios y protección de contraseñas mediante algoritmos de hashing. |
| **Especificación** | POST /api/registro: recibe usuario y password, genera un hash con Bcrypt, almacena únicamente el hash y responde HTTP 201.POST /api/login: recibe usuario y password, verifica las credenciales utilizando Bcrypt y responde HTTP 200 si son correctas o HTTP 401 si son incorrectas.|
| **Output** | Entregar el archivo app.py con los modelos de entrada y salida, las rutas de registro e inicio de sesión, el hashing seguro con Bcrypt y el manejo de errores HTTP. La implementación debe cumplir los casos de verificación del cuaderno. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'El registro responde sin filtrar la contraseña',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'json_contiene': {'usuario': 'ana'},
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'El login correcto autentica',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'autenticado': True}},
 {'descripcion': 'La contraseña incorrecta se rechaza',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'incorrecta'},
  'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. El registro responde sin filtrar la contraseña
✅ 02. El login correcto autentica
✅ 03. La contraseña incorrecta se rechaza
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["bcrypt|passlib"], descripcion="Se usa un algoritmo de hashing para contraseñas"))
resumen(resultados)


✅ Se usa un algoritmo de hashing para contraseñas
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion14.md` a `auditorias/auditoria_sesion14.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?R.- Lo que más me costó fue implementar correctamente la verificación de las contraseñas con Bcrypt, porque tuve que comprender cómo generar el hash y después comprobar la contraseña sin necesidad de recuperarla.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?R.- Decidí utilizar un diccionario para almacenar temporalmente los usuarios y sus hashes. También agregué una validación para evitar que un usuario repetido sobrescriba su contraseña y responder con el código HTTP 409.
   - ¿Qué error detectó la verificación que tú no habías previsto?R.- Decidí utilizar un diccionario para almacenar temporalmente los usuarios y sus hashes. También agregué una validación para evitar que un usuario repetido sobrescriba su contraseña y responder con el código HTTP 409.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
